# FIGNN — Final Stage 1 and Stage 2 Demonstration

This notebook is a **read-only demonstration** of the final FIGNN evaluation pipeline.

No training, retraining, checkpoint generation, result generation, or file saving is performed.

## Stage 1 — Unseen-subject transferability

For a representative held-out subject, we:

- load the corresponding frozen LOSO FIGNN encoder,
- generate latent embeddings,
- visualize the learned representation,
- compare the subject's workload trajectory with the population trajectory, and
- demonstrate trajectory-shape transferability using Procrustes alignment.

The final all-29-subject Stage 1 results are loaded from the completed LOSO evaluation artifacts.

## Stage 2 — Unseen-subject calibration

We demonstrate two complementary results:

1. **Direct 64-D latent geometry**
   - distance between 0-back and 1-back,
   - distance between 1-back and 2-back,
   - distance between 0-back and 2-back.

2. **Calibration comparison**
   - population-only estimation of an unseen subject's 1-back embedding,
   - REST + known 2-back calibrated estimation,
   - comparison of 64-D Euclidean prediction errors.

All final aggregate results are loaded from the completed evaluation files.

In [25]:
# ============================================================
# 0. SETUP
# ============================================================

import os
import sys
import json
import pickle
import copy

import numpy as np
import pandas as pd
import torch

import plotly.graph_objects as go
import plotly.express as px

from scipy.spatial import procrustes
from sklearn.decomposition import PCA
from torch_geometric.loader import DataLoader

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = os.path.abspath("..")

print("Project root:", PROJECT_ROOT)

GRAPHS_PATH = (
    "/lp-dev/24BCE1793/data/processed/"
    "all_graphs_29sub.pkl"
)

LOSO_ROOT = (
    "/lp-dev/24BCE1793/outputs/loso"
)

FINAL_NORMALISATION = (
    "/lp-dev/24BCE1793/outputs/"
    "checkpoints_full/normalisation_stats.pt"
)

STAGE1_RESULTS_ROOT = (
    "/lp-dev/24BCE1793/outputs/"
    "evaluation_loso_all29"
)

STAGE2_DISTANCES_PATH = (
    "/lp-dev/24BCE1793/outputs/"
    "stage2_centroid_64D_distances.csv"
)

STAGE2_CALIBRATION_PATH = (
    "/lp-dev/24BCE1793/outputs/"
    "stage2_calibration_comparison_64D.csv"
)


# ------------------------------------------------------------
# Representative subject for interactive demonstration
# ------------------------------------------------------------

DEMO_SUBJECT = 17

TASK_LABELS = [0, 1, 2]

LABEL_NAMES = {
    0: "0-back",
    1: "1-back",
    2: "2-back"
}


# ------------------------------------------------------------
# Imports from project
# ------------------------------------------------------------

sys.path.insert(0, PROJECT_ROOT)

from src.models.model import FIGNNEncoder


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)
print("Demo subject:", DEMO_SUBJECT)

Project root: /home/nvidia/24BCE1793/fignn_cni__project
Using device: cuda
Demo subject: 17


In [26]:
# ============================================================
# 1. HELPERS
# ============================================================

def get_subject(graph):
    return int(graph.subject)


def get_label(graph):
    return int(graph.y.item())


def normalize_graphs(graphs, mean, std):

    normalized = []

    for graph in graphs:

        g = copy.copy(graph)

        g.x = (
            g.x.clone() - mean
        ) / std

        normalized.append(g)

    return normalized


@torch.no_grad()
def encode_graphs(
    model,
    graphs,
    mean,
    std,
    device,
    batch_size=128
):

    normalized = normalize_graphs(
        graphs,
        mean,
        std
    )

    loader = DataLoader(
        normalized,
        batch_size=batch_size,
        shuffle=False
    )

    embeddings = []
    labels = []

    model.eval()

    for batch in loader:

        batch = batch.to(device)

        z = model(batch)

        embeddings.append(
            z.detach().cpu().numpy()
        )

        labels.append(
            batch.y.view(-1).cpu().numpy()
        )

    return (
        np.vstack(embeddings),
        np.concatenate(labels)
    )


def condition_centroids(
    embeddings,
    labels
):

    centroids = {}

    for label in TASK_LABELS:

        centroids[label] = (
            embeddings[labels == label]
            .mean(axis=0)
        )

    return centroids


def trajectory_array(centroids):

    return np.vstack([
        centroids[0],
        centroids[1],
        centroids[2]
    ])

In [27]:
# ============================================================
# 2. LOAD FINAL GRAPH DATA
# ============================================================

print("Loading graphs...")

with open(GRAPHS_PATH, "rb") as f:
    all_graphs = pickle.load(f)

print("Total graphs:", len(all_graphs))

subjects = sorted(
    set(get_subject(g) for g in all_graphs)
)

print("Subjects:", subjects)

task_graphs = [
    g for g in all_graphs
    if get_label(g) in TASK_LABELS
]

print("Task graphs:", len(task_graphs))

Loading graphs...
Total graphs: 27666
Subjects: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
Task graphs: 25056


In [28]:
# ============================================================
# 3. LOAD REPRESENTATIVE FROZEN LOSO MODEL
# ============================================================

CHECKPOINT = os.path.join(
    LOSO_ROOT,
    f"subject_{DEMO_SUBJECT:02d}",
    "fignn_final.pth"
)

print("Checkpoint:")
print(CHECKPOINT)

model = FIGNNEncoder().to(device)

state_dict = torch.load(
    CHECKPOINT,
    map_location=device
)

model.load_state_dict(state_dict)

model.eval()

print("Frozen LOSO model loaded successfully.")

Checkpoint:
/lp-dev/24BCE1793/outputs/loso/subject_17/fignn_final.pth
Frozen LOSO model loaded successfully.


In [29]:
# ============================================================
# 4. LOAD FINAL NORMALIZATION STATISTICS
# ============================================================

stats = torch.load(
    FINAL_NORMALISATION,
    map_location="cpu"
)

mean = stats["mean"].float()

std = (
    stats["std"]
    .float()
    .clamp(min=1e-8)
)

print("Normalization loaded.")
print("Feature dimension:", len(mean))

Normalization loaded.
Feature dimension: 4


In [30]:
# ============================================================
# 5. CONSTRUCT DEMO LOSO SPLIT
# ============================================================

train_graphs = [
    g for g in task_graphs
    if get_subject(g) != DEMO_SUBJECT
]

test_graphs = [
    g for g in task_graphs
    if get_subject(g) == DEMO_SUBJECT
]

print(
    f"Training subjects: "
    f"{len(set(get_subject(g) for g in train_graphs))}"
)

print(
    f"Held-out subject graphs: "
    f"{len(test_graphs)}"
)

Training subjects: 28
Held-out subject graphs: 864


In [31]:
# ============================================================
# 6. GENERATE DEMO EMBEDDINGS
# ============================================================

print("Encoding held-out subject...")

z_test, y_test = encode_graphs(
    model=model,
    graphs=test_graphs,
    mean=mean,
    std=std,
    device=device
)

print("Held-out embeddings:", z_test.shape)

Encoding held-out subject...
Held-out embeddings: (864, 64)


In [ ]:
# ============================================================
# 7. LATENT SPACE VISUALIZATION — MDS
# ============================================================

from sklearn.manifold import MDS


# ------------------------------------------------------------
# True 64-D condition centroids
# ------------------------------------------------------------

centroids = np.vstack([
    z_test[y_test == 0].mean(axis=0),
    z_test[y_test == 1].mean(axis=0),
    z_test[y_test == 2].mean(axis=0)
])


# ------------------------------------------------------------
# Embed trials + centroids together
# ------------------------------------------------------------

z_mds_input = np.vstack([
    z_test,
    centroids
])


mds = MDS(
    n_components=2,
    metric=True,
    random_state=42,
    n_init=4,
    max_iter=300
)

z_2d = mds.fit_transform(z_mds_input)


# ------------------------------------------------------------
# Split trial and centroid coordinates
# ------------------------------------------------------------

sample_coords = z_2d[:-3]
centroid_coords = z_2d[-3:]


# ------------------------------------------------------------
# Trial-level dataframe
# ------------------------------------------------------------

plot_df = pd.DataFrame({
    "MDS 1": sample_coords[:, 0],
    "MDS 2": sample_coords[:, 1],
    "Condition": [
        LABEL_NAMES[int(y)]
        for y in y_test
    ]
})


# ------------------------------------------------------------
# Individual trial embeddings
# ------------------------------------------------------------

fig = px.scatter(
    plot_df,
    x="MDS 1",
    y="MDS 2",
    color="Condition",
    opacity=0.45
)


fig.update_traces(
    marker=dict(size=5)
)


# ------------------------------------------------------------
# Centroid trajectory
#
# The line shows:
# 0-back → 1-back → 2-back
# ------------------------------------------------------------

fig.add_trace(
    go.Scatter(
        x=centroid_coords[:, 0],
        y=centroid_coords[:, 1],
        mode="lines+markers",
        name="Condition trajectory",
        marker=dict(
            size=13,
            symbol="diamond"
        ),
        line=dict(
            width=3
        ),
        text=[
            "0-back centroid",
            "1-back centroid",
            "2-back centroid"
        ],
        hovertemplate=(
            "%{text}<br>"
            "MDS 1: %{x:.4f}<br>"
            "MDS 2: %{y:.4f}"
            "<extra></extra>"
        )
    )
)


# ------------------------------------------------------------
# Paper-style layout
# ------------------------------------------------------------

fig.update_layout(
    template="plotly_white",

    title=dict(
        text=(
            "MDS Projection of the 64-Dimensional FIGNN Latent Space"
        ),
        x=0.5,
        xanchor="center"
    ),

    height=700,
    width=850,

    legend=dict(
        title="Condition",
        x=1.01,
        y=1
    ),

    margin=dict(
        l=60,
        r=160,
        t=80,
        b=60
    )
)


# ------------------------------------------------------------
# Equal physical scale
#
# Important because this is a distance-preserving visualization.
# ------------------------------------------------------------

fig.update_yaxes(
    title="MDS dimension 2",
    scaleanchor="x",
    scaleratio=1
)

fig.update_xaxes(
    title="MDS dimension 1"
)


fig.show()

/lp-dev/24BCE1793/fignn_env/lib/python3.13/site-packages/sklearn/manifold/_mds.py:735: FutureWarning: The default value of `init` will change from 'random' to 'classical_mds' in 1.10. To suppress this warning, provide some value of `init`.
  warnings.warn(
/lp-dev/24BCE1793/fignn_env/lib/python3.13/site-packages/sklearn/manifold/_mds.py:760: FutureWarning: Use metric_mds=True instead of metric=True. The support for metric={True/False} will be dropped in 1.10.
  warnings.warn(


In [ ]:
# ============================================================
# 8. LEARNED FREQUENCY-BAND IMPORTANCE
# ============================================================

band_weights = (
    model.get_band_importance()
)

band_df = pd.DataFrame({
    "Band": list(band_weights.keys()),
    "Importance": list(
        band_weights.values()
    )
})

display(band_df.round(6))

fig = px.bar(
    band_df,
    x="Band",
    y="Importance",
    title="Learned Frequency-Band Importance"
)

fig.update_layout(
    template="plotly_white",
    height=500
)

fig.show()

In [ ]:
# ============================================================
# 9. BUILD POPULATION WORKLOAD TRAJECTORY
# ============================================================

training_subjects = sorted(
    set(
        get_subject(g)
        for g in train_graphs
    )
)

population_centroids = []

for label in TASK_LABELS:

    subject_centroids = []

    for subject in training_subjects:

        subject_graphs = [
            g for g in train_graphs
            if get_subject(g) == subject
            and get_label(g) == label
        ]

        z_subject, _ = encode_graphs(
            model=model,
            graphs=subject_graphs,
            mean=mean,
            std=std,
            device=device
        )

        subject_centroids.append(
            z_subject.mean(axis=0)
        )

    population_centroids.append(
        np.mean(
            subject_centroids,
            axis=0
        )
    )

population_centroids = np.vstack(
    population_centroids
)

subject_centroids = trajectory_array(
    condition_centroids(
        z_test,
        y_test
    )
)

print(
    "Population trajectory shape:",
    population_centroids.shape
)

print(
    "Subject trajectory shape:",
    subject_centroids.shape
)

In [ ]:
# ============================================================
# 10. POPULATION vs SUBJECT TRAJECTORY
# ============================================================

combined = np.vstack([
    population_centroids,
    subject_centroids
])


# ------------------------------------------------------------
# Joint projection
#
# PCA is fitted ONCE to all six 64-D centroids so population
# and held-out subject trajectories share the same coordinate
# system.
# ------------------------------------------------------------

trajectory_pca = PCA(
    n_components=2
)

combined_2d = trajectory_pca.fit_transform(
    combined
)

population_2d = combined_2d[:3]

subject_2d = combined_2d[3:]


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig = go.Figure()


# Population trajectory
fig.add_trace(
    go.Scatter(
        x=population_2d[:, 0],
        y=population_2d[:, 1],

        mode="lines+markers+text",

        text=[
            "0-back",
            "1-back",
            "2-back"
        ],

        textposition="top center",

        name="Population",

        marker=dict(
            size=10
        ),

        line=dict(
            width=3
        )
    )
)


# Held-out subject trajectory
fig.add_trace(
    go.Scatter(
        x=subject_2d[:, 0],
        y=subject_2d[:, 1],

        mode="lines+markers+text",

        text=[
            "0-back",
            "1-back",
            "2-back"
        ],

        textposition="bottom center",

        name=(
            f"Held-out Subject {DEMO_SUBJECT}"
        ),

        marker=dict(
            size=10
        ),

        line=dict(
            width=3,
            dash="dash"
        )
    )
)


# ------------------------------------------------------------
# Layout
# ------------------------------------------------------------

fig.update_layout(
    title=(
        "Population and Held-out Subject "
        "Workload Trajectories"
    ),

    xaxis_title=(
        "Projected latent dimension 1"
    ),

    yaxis_title=(
        "Projected latent dimension 2"
    ),

    template="plotly_white",

    height=600,

    width=900,

    legend=dict(
        title="Trajectory"
    )
)


# ------------------------------------------------------------
# Equal geometric scaling
# ------------------------------------------------------------

fig.update_yaxes(
    scaleanchor="x",
    scaleratio=1
)


fig.show()

In [ ]:
# ============================================================
# 11. PROCRUSTES TRAJECTORY SHAPE COMPARISON
# ============================================================

from scipy.spatial import procrustes
from sklearn.decomposition import PCA
import plotly.graph_objects as go
import numpy as np


# ------------------------------------------------------------
# 1. Procrustes alignment in full 64-D latent space
# ------------------------------------------------------------

population_mtx, subject_mtx, disparity = procrustes(
    population_centroids,
    subject_centroids
)

shape_similarity = 1.0 - disparity

print(f"Representative subject: {DEMO_SUBJECT}")
print(f"Procrustes disparity: {disparity:.6f}")
print(f"Shape similarity: {shape_similarity:.6f}")


# ------------------------------------------------------------
# 2. PCA ONLY for 2-D visualization
# ------------------------------------------------------------

combined = np.vstack([
    population_mtx,
    subject_mtx
])

pca = PCA(n_components=2)

combined_2d = pca.fit_transform(combined)

population_2d = combined_2d[:3]
subject_2d = combined_2d[3:]


print(
    f"PCA variance shown: "
    f"{pca.explained_variance_ratio_.sum():.2%}"
)


# ------------------------------------------------------------
# 3. Condition labels
# ------------------------------------------------------------

conditions = [
    "0-back",
    "1-back",
    "2-back"
]


# ------------------------------------------------------------
# 4. Plot
# ------------------------------------------------------------

fig = go.Figure()


# ============================================================
# Population trajectory
# ============================================================

fig.add_trace(
    go.Scatter(
        x=population_2d[:, 0],
        y=population_2d[:, 1],

        mode="lines+markers+text",

        text=conditions,

        # Put population labels above/right of points
        textposition=[
            "top left",
            "top center",
            "top center"
        ],

        name="Population",

        marker=dict(
            size=10
        ),

        line=dict(
            width=3
        ),

        hovertemplate=(
            "Population"
            "<br>%{text}"
            "<br>PC1 = %{x:.3f}"
            "<br>PC2 = %{y:.3f}"
            "<extra></extra>"
        )
    )
)


# ============================================================
# Subject trajectory
# ============================================================

fig.add_trace(
    go.Scatter(
        x=subject_2d[:, 0],
        y=subject_2d[:, 1],

        mode="lines+markers+text",

        text=conditions,

        # Put subject labels below/right of points
        textposition=[
            "bottom right",
            "bottom center",
            "bottom center"
        ],

        name=f"Subject {DEMO_SUBJECT}",

        marker=dict(
            size=10
        ),

        line=dict(
            width=3,
            dash="dash"
        ),

        hovertemplate=(
            f"Subject {DEMO_SUBJECT}"
            "<br>%{text}"
            "<br>PC1 = %{x:.3f}"
            "<br>PC2 = %{y:.3f}"
            "<extra></extra>"
        )
    )
)


# ============================================================
# Layout
# ============================================================

fig.update_layout(
    template="plotly_white",

    width=850,
    height=600,

    margin=dict(
        l=70,
        r=30,
        t=65,
        b=65
    ),

    font=dict(
        family="Times New Roman",
        size=14
    ),

    title=(
        f"Procrustes-Aligned Workload Trajectories "
        f"(Subject {DEMO_SUBJECT})"
    ),

    xaxis_title="PC 1",
    yaxis_title="PC 2",

    legend=dict(
        title="Trajectory"
    )
)


# Preserve geometric aspect ratio
fig.update_yaxes(
    scaleanchor="x",
    scaleratio=1
)


fig.show()


# ============================================================
# Export
# ============================================================

fig.write_html(
    "fig_procrustes_subject17.html"
)

try:

    fig.write_image(
        "fig_procrustes_subject17.pdf",
        width=850,
        height=600,
        scale=2
    )

    fig.write_image(
        "fig_procrustes_subject17.png",
        width=1700,
        height=1200,
        scale=1
    )

    print("\nSaved Procrustes figure.")

except Exception as e:

    print(f"\nExport failed: {e}")
    print("If required: pip install -U kaleido")

In [ ]:
# ============================================================
# 12. FINAL STAGE 1 — ALL 29 LOSO RESULTS
# ============================================================

STAGE1_SUBJECT_METRICS = os.path.join(
    STAGE1_RESULTS_ROOT,
    "loso_all29_subject_metrics.csv"
)

STAGE1_REPORT = os.path.join(
    STAGE1_RESULTS_ROOT,
    "loso_all29_report.json"
)


# ------------------------------------------------------------
# Load final Stage 1 results
# ------------------------------------------------------------

stage1_df = pd.read_csv(
    STAGE1_SUBJECT_METRICS
)

with open(
    STAGE1_REPORT,
    "r"
) as f:

    stage1_report = json.load(f)


print("Final all-29 LOSO evaluation loaded.")
print(
    f"Subjects evaluated: "
    f"{stage1_report['n_subjects']}"
)


# ============================================================
# KEY STAGE 1 FINDINGS
# ============================================================

metrics = stage1_report["metrics"]
counts = stage1_report["counts"]

n_subjects = stage1_report[
    "n_subjects"
]


summary_df = pd.DataFrame([

    {
        "Finding":
            "Mean Procrustes trajectory similarity",

        "Value":
            round(
                metrics[
                    "procrustes_similarity"
                ]["mean"],
                3
            ),

        "95% CI":
            (
                f"{metrics['procrustes_similarity']['bootstrap_95_low']:.3f}"
                " – "
                f"{metrics['procrustes_similarity']['bootstrap_95_high']:.3f}"
            )
    },


    {
        "Finding":
            "Subjects with 0B–1B < 0B–2B",

        "Value":
            (
                f"{counts['monotonic_d01_lt_d02']} / "
                f"{n_subjects}"
            ),

        "95% CI":
            ""
    },


    {
        "Finding":
            "Subjects with 1B between 0B and 2B",

        "Value":
            (
                f"{counts['oneback_between']} / "
                f"{n_subjects}"
            ),

        "95% CI":
            ""
    },


    {
        "Finding":
            "Mean trajectory path efficiency",

        "Value":
            round(
                metrics[
                    "path_efficiency"
                ]["mean"],
                3
            ),

        "95% CI":
            (
                f"{metrics['path_efficiency']['bootstrap_95_low']:.3f}"
                " – "
                f"{metrics['path_efficiency']['bootstrap_95_high']:.3f}"
            )
    }

])


print(
    "\n"
    "STAGE 1 — KEY FINDINGS\n"
)

display(
    summary_df
)


# ============================================================
# SUBJECT-LEVEL RESULTS
# ============================================================

subject_columns = [

    "subject",

    # Latent separation
    "silhouette",

    # Trajectory structure
    "monotonicity",
    "oneback_between",

    # Population ↔ subject trajectory similarity
    "procrustes_similarity",

    # Additional evaluation metrics
    "knn5_accuracy",
    "path_efficiency",
    "angle_deg"

]


# Keep only columns that actually exist
subject_columns = [

    col
    for col in subject_columns

    if col in stage1_df.columns
]


subject_results = (

    stage1_df[
        subject_columns
    ]

    .sort_values(
        "subject"
    )

    .reset_index(
        drop=True
    )

)


print(
    "\n"
    "SUBJECT-LEVEL LOSO RESULTS\n"
)

display(
    subject_results.round(4)
)


# ============================================================
# EVALUATION NOTE
# ============================================================

print(
    "\nEvaluation note: All Stage 1 metrics were computed "
    "using fold-specific frozen LOSO checkpoints. The held-out "
    "subject was not used to select a checkpoint."
)

In [ ]:
# ============================================================
# 12A. FINAL STAGE 1 — CORE METRICS VISUALIZATION
# ============================================================

import plotly.graph_objects as go
from plotly.subplots import make_subplots


# ------------------------------------------------------------
# Use authoritative all-29 LOSO results already loaded
# in Cell 13.
# ------------------------------------------------------------

plot_df = (
    stage1_df[
        [
            "subject",
            "procrustes_similarity",
            "direction_cosine"
        ]
    ]
    .dropna()
    .sort_values("subject")
    .copy()
)


# ------------------------------------------------------------
# Summary statistics
# ------------------------------------------------------------

mean_procrustes = (
    plot_df["procrustes_similarity"]
    .mean()
)

mean_direction = (
    plot_df["direction_cosine"]
    .mean()
)


print(
    f"Mean Procrustes similarity: "
    f"{mean_procrustes:.6f}"
)

print(
    f"Mean subject-population direction cosine: "
    f"{mean_direction:.6f}"
)


# ============================================================
# FIGURE
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Procrustes similarity",
        "Subject–population direction"
    ],
    horizontal_spacing=0.12
)


# ------------------------------------------------------------
# (a) Procrustes similarity
# ------------------------------------------------------------

fig.add_trace(
    go.Scatter(
        x=plot_df["subject"],
        y=plot_df["procrustes_similarity"],
        mode="markers",

        marker=dict(
            size=8
        ),

        name="Procrustes",
        showlegend=False,

        hovertemplate=(
            "Subject %{x}"
            "<br>Similarity = %{y:.4f}"
            "<extra></extra>"
        )
    ),
    row=1,
    col=1
)


# Mean reference
fig.add_hline(
    y=mean_procrustes,
    line_dash="dash",
    annotation_text=(
        f"Mean = {mean_procrustes:.3f}"
    ),
    annotation_position="top left",
    row=1,
    col=1
)


# ------------------------------------------------------------
# (b) Subject–population direction cosine
# ------------------------------------------------------------

fig.add_trace(
    go.Scatter(
        x=plot_df["subject"],
        y=plot_df["direction_cosine"],
        mode="markers",

        marker=dict(
            size=8
        ),

        name="Direction cosine",
        showlegend=False,

        hovertemplate=(
            "Subject %{x}"
            "<br>Direction cosine = %{y:.4f}"
            "<extra></extra>"
        )
    ),
    row=1,
    col=2
)


# Zero reference
fig.add_hline(
    y=0,
    line_dash="dash",
    row=1,
    col=2
)


# ============================================================
# AXES
# ============================================================

fig.update_xaxes(
    title_text="Subject",
    tickmode="array",
    tickvals=[1, 5, 10, 15, 20, 25, 29],
    row=1,
    col=1
)

fig.update_xaxes(
    title_text="Subject",
    tickmode="array",
    tickvals=[1, 5, 10, 15, 20, 25, 29],
    row=1,
    col=2
)

fig.update_yaxes(
    title_text="Procrustes similarity",
    range=[
        max(0.0, plot_df["procrustes_similarity"].min() - 0.05),
        1.02
    ],
    row=1,
    col=1
)

fig.update_yaxes(
    title_text="Direction cosine",
    range=[-1.05, 1.05],
    row=1,
    col=2
)


# ============================================================
# IEEE-LIKE CLEAN LAYOUT
# ============================================================

fig.update_layout(
    template="plotly_white",

    width=1000,
    height=420,

    margin=dict(
        l=70,
        r=30,
        t=65,
        b=65
    ),

    font=dict(
        family="Times New Roman",
        size=14
    ),

    title=None,

    showlegend=False
)


# ------------------------------------------------------------
# Export
# ------------------------------------------------------------

fig.show()

fig.write_html(
    "fig_stage1_core_metrics.html"
)

try:
    fig.write_image(
        "fig_stage1_core_metrics.pdf",
        width=1000,
        height=420,
        scale=2
    )

    fig.write_image(
        "fig_stage1_core_metrics.png",
        width=2000,
        height=840,
        scale=1
    )

    print(
        "\nSaved:"
        "\n  fig_stage1_core_metrics.pdf"
        "\n  fig_stage1_core_metrics.png"
        "\n  fig_stage1_core_metrics.html"
    )

except Exception as e:

    print(
        "\nPDF/PNG export failed."
        "\nInstall kaleido if needed:"
        "\n  pip install -U kaleido"
        f"\n\nError: {e}"
    )

# Stage 2 — Direct 64-D geometry and calibration

Stage 2 evaluates personalization directly in the original FIGNN latent space.

Unlike the earlier trajectory-position experiment, the final evaluation does not use PCA, MDS, or another dimensionality reduction method for the error calculation.

All distances and prediction errors are computed directly in the 64-dimensional latent space.

In [ ]:
# ============================================================
# 13. STAGE 2A — DIRECT 64-D CENTROID DISTANCES
# ============================================================

distance_df = pd.read_csv(
    STAGE2_DISTANCES_PATH
)

display(
    distance_df.round(6)
)


distance_summary = (
    distance_df[
        [
            "d_0B_1B",
            "d_1B_2B",
            "d_0B_2B"
        ]
    ]
    .agg(
        ["mean", "median", "std"]
    )
)

display(
    distance_summary.round(6)
)


count_01_lt_02 = (
    distance_df["d_0B_1B"]
    <
    distance_df["d_0B_2B"]
).sum()

count_12_lt_02 = (
    distance_df["d_1B_2B"]
    <
    distance_df["d_0B_2B"]
).sum()


print(
    f"Subjects where "
    f"0B–1B < 0B–2B: "
    f"{count_01_lt_02} / "
    f"{len(distance_df)}"
)

print(
    f"Subjects where "
    f"1B–2B < 0B–2B: "
    f"{count_12_lt_02} / "
    f"{len(distance_df)}"
)

In [ ]:
# ============================================================
# 14. 64-D CONDITION DISTANCE VISUALIZATION
# ============================================================

distance_long = distance_df.melt(
    id_vars="Subject",
    value_vars=[
        "d_0B_1B",
        "d_1B_2B",
        "d_0B_2B"
    ],
    var_name="Distance",
    value_name="64-D Euclidean Distance"
)

distance_long["Distance"] = (
    distance_long["Distance"]
    .replace({
        "d_0B_1B": "0-back ↔ 1-back",
        "d_1B_2B": "1-back ↔ 2-back",
        "d_0B_2B": "0-back ↔ 2-back"
    })
)


fig = px.box(
    distance_long,
    x="Distance",
    y="64-D Euclidean Distance",
    points="all",
    title=(
        "Direct 64-D Condition "
        "Centroid Distances Across LOSO Subjects"
    )
)

fig.update_layout(
    template="plotly_white",
    height=600
)

fig.show()

In [ ]:
# ============================================================
# 14A. LOAD FINAL STAGE 2 CALIBRATION RESULTS
# ============================================================

calibration_df = pd.read_csv(
    STAGE2_CALIBRATION_PATH
)

print("Final Stage 2 calibration results loaded.")
print(
    f"Subjects evaluated: {len(calibration_df)}"
)

print(
    "\nColumns:"
)

print(
    calibration_df.columns.tolist()
)

display(
    calibration_df.head()
)

In [ ]:
# ============================================================
# 15. FINAL STAGE 2 — CALIBRATION COMPARISON
# ============================================================

# ------------------------------------------------------------
# Use actual Stage 2 result columns
# ------------------------------------------------------------

baseline_col = "baseline_error_64D"
calibrated_col = "calibrated_error_64D"
improvement_col = "percent_improvement"
helped_col = "calibration_helped"


# ------------------------------------------------------------
# Aggregate results
# ------------------------------------------------------------

baseline_mean = calibration_df[baseline_col].mean()

calibrated_mean = calibration_df[calibrated_col].mean()

mean_improvement = calibration_df[improvement_col].mean()

helped_count = calibration_df[helped_col].sum()

n_subjects = len(calibration_df)


# ------------------------------------------------------------
# Stage 2 summary
# ------------------------------------------------------------

stage2_summary = pd.DataFrame([
    {
        "Finding": "Mean population-only prediction error",
        "Value": baseline_mean
    },
    {
        "Finding": "Mean calibrated prediction error",
        "Value": calibrated_mean
    },
    {
        "Finding": "Mean percentage improvement",
        "Value": f"{mean_improvement:.2f}%"
    },
    {
        "Finding": "Subjects improved by calibration",
        "Value": f"{helped_count} / {n_subjects}"
    }
])


print("\nSTAGE 2 — CALIBRATION COMPARISON\n")

display(
    stage2_summary
)


# ------------------------------------------------------------
# Interpretation
# ------------------------------------------------------------

print(
    "\nComparison: the population-only baseline predicts the "
    "held-out subject's 1-back latent centroid without any "
    "subject-specific information. The calibrated prediction "
    "uses the held-out subject's REST (0-back) and 2-back "
    "centroids as personal calibration anchors."
)

print(
    f"\nMean baseline error:   {baseline_mean:.6f}"
)

print(
    f"Mean calibrated error: {calibrated_mean:.6f}"
)

print(
    f"Calibration improved {helped_count} of "
    f"{n_subjects} unseen subjects."
)

In [ ]:
# ============================================================
# 16. BASELINE vs CALIBRATED ERROR — FINAL FIGURE
# ============================================================

comparison_long = pd.DataFrame({
    "Subject": np.concatenate([
        calibration_df["Subject"].values,
        calibration_df["Subject"].values
    ]),

    "Method": (
        ["Population-only"] * len(calibration_df)
        +
        ["REST + 2-back calibrated"] * len(calibration_df)
    ),

    "64-D Error": np.concatenate([
        calibration_df[baseline_col].values,
        calibration_df[calibrated_col].values
    ])
})


fig = px.box(
    comparison_long,
    x="Method",
    y="64-D Error",
    points="all"
)


fig.update_traces(
    jitter=0.18,
    pointpos=0,
    marker=dict(
        size=6
    )
)


fig.update_layout(
    template="plotly_white",

    width=750,
    height=500,

    margin=dict(
        l=70,
        r=30,
        t=35,
        b=75
    ),

    font=dict(
        family="Times New Roman",
        size=14
    ),

    title=None,

    xaxis_title=None,

    yaxis_title=(
        "64-D Euclidean prediction error"
    )
)


fig.show()


try:

    fig.write_image(
        "fig_stage2_calibration_error.pdf",
        width=750,
        height=500,
        scale=2
    )

    fig.write_image(
        "fig_stage2_calibration_error.png",
        width=1500,
        height=1000,
        scale=1
    )

    print(
        "\nSaved Stage 2 figure."
    )

except Exception as e:

    print(
        f"\nExport failed: {e}"
    )

In [ ]:
# ============================================================
# 17. SUBJECT-WISE CALIBRATION EFFECT
# ============================================================

fig = go.Figure()

for _, row in calibration_df.iterrows():

    fig.add_trace(
        go.Scatter(
            x=[
                "Population-only",
                "REST + 2-back calibrated"
            ],

            y=[
                row[baseline_col],
                row[calibrated_col]
            ],

            mode="lines+markers",

            showlegend=False,

            hovertemplate=(
                f"Subject {int(row['Subject'])}"
                "<br>%{x}"
                "<br>Error = %{y:.6f}"
                "<extra></extra>"
            )
        )
    )


fig.update_layout(
    title=(
        "Subject-wise Effect of Personal Calibration"
    ),

    yaxis_title=(
        "64-D Euclidean Prediction Error"
    ),

    template="plotly_white",

    height=650
)

fig.show()

# Final Demonstration Summary

## Stage 1

The frozen FIGNN encoder was evaluated under a 29-fold leave-one-subject-out setting.

The demonstration visualizes a representative unseen subject, while the completed all-subject evaluation provides the quantitative evidence for latent trajectory transferability.

Procrustes comparison evaluates similarity of workload trajectory shape independently of the particular coordinate orientation or scale used for visualization.

## Stage 2

Direct analysis of condition centroids in the original 64-dimensional latent space shows that the learned geometry generally preserves the expected relationship between workload conditions.

The calibration experiment then evaluates a more practically relevant question:

> Can subject-specific REST and known 2-back anchors improve estimation of an unseen subject's 1-back latent state?

Population-only and calibrated estimates are evaluated against the true 1-back centroid directly using 64-D Euclidean error.

This provides a quantitative test of whether minimal personal calibration improves workload-state estimation for unseen subjects.